# Data cleaning & preparation :

In [1260]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [1261]:
orders = pd.read_csv("../Data/Raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../Data/Raw/olist_order_items_dataset.csv")
customers = pd.read_csv("../Data/Raw/olist_customers_dataset.csv")
payments = pd.read_csv("../Data/Raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../Data/Raw/olist_order_reviews_dataset.csv")
sellers =pd.read_csv("../Data/Raw/olist_sellers_dataset.csv")
products = pd.read_csv("../Data/Raw/olist_products_dataset.csv")
product_name_translation = pd.read_csv("../Data/Raw/product_category_name_translation.csv")
geolocation = pd.read_csv("../Data/Raw/olist_geolocation_dataset.csv")


In [1262]:
#dataframes copies for cleaning

orders_clean = orders.copy()
order_items_clean = order_items.copy()
customers_clean = customers.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
sellers_clean = sellers.copy()
products_clean = products.copy()
geolocation_clean = geolocation.copy()

## orders Table :

### Missing values :

In [ ]:
#Missing Values overview

df = orders_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum() ,
    "missing_percent" : df.isna().mean() * 100
})

missing_overview = missing_overview[missing_overview["missing_count"] > 0].sort_values(by="missing_percent",ascending=False)
display(missing_overview)

,missing_count,missing_percent
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899


In [1264]:
#investigating missing values

#check order status of missing values
df = orders_clean

missing_customer_delivery = df[df["order_delivered_customer_date"].isna()]

print("--- order status of missing customer delivery ---")
print(missing_customer_delivery["order_status"].value_counts(dropna=False))

missing_carrier_delivery = df[df["order_delivered_carrier_date"].isna()]

print("\n--- order status of missing carrier delivery ---")
print(missing_carrier_delivery["order_status"].value_counts(dropna=False))

missing_approved = df[df["order_approved_at"].isna()]

print("\n--- order status of missing approved ---")
print(missing_approved["order_status"].value_counts(dropna=False))


--- order status of missing customer delivery ---
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

--- order status of missing carrier delivery ---
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

--- order status of missing approved ---
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64


In [1343]:
#checking inconsistencies 

#for customer delivery
display(missing_customer_delivery[missing_customer_delivery["order_status"] == "delivered"].head())

#for carrier delivery
display(missing_carrier_delivery[missing_carrier_delivery["order_status"] == "delivered"].head())

#for approved
display(missing_approved[missing_approved["order_status"] == "delivered"].head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00


#### cleaning decision :

No rows will be deleted, and no missing timestamps will be imputed at this stage, because Most missing delivery timestamps belong to orders whose statuses indicate that they may not have reached the relevent delivery stage.

The identified inconsistencies will be documented for further investigation. inconsistencies are those that have wrong status for example in this table some orders have delivered status but the customer delivered timestamp did not recorded.
but we couldn't drop all those inconsistencies because we would lose some important data, so that's why we documented them

### duplicates :

In [1266]:
df = orders_clean

#checking total duplicates
print("Exact duplicates:", df.duplicated().sum())

print(
    "Duplicate order_id:",
    df["order_id"].duplicated().sum()
)

Exact duplicates: 0
Duplicate order_id: 0


#### Duplicate Records

**Investigation:** Exact duplicate rows and duplicate `order_id` values were checked.

**Decision:** No duplicate records were identified based on these checks.

**Action:** No rows were removed.

**Validation:** The duplicate checks confirmed that the identified conditions were not present in the current dataset.

### Data types :

In [1267]:
df = orders_clean

display(df.dtypes)

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [1268]:
# Find the orders that are null after converting to datetime but they weren't null before that

parsed = pd.to_datetime(
    df["order_delivered_customer_date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

problem_mask = df["order_delivered_customer_date"].notna() & parsed.isna()

print("Number of problematic values:", problem_mask.sum())

display(df.loc[problem_mask, ["order_id", "order_delivered_customer_date"]])



Number of problematic values: 2


,order_id,order_delivered_customer_date
33,d887b52c6516beb39e8cd44a5f8b60f7,2018-02-22 007:55
169,b2f3763408103bc7173bb36ed02dbeea,2018-06-18:0 23:07:33


In [1269]:
# Fix porblematic values in order_delivered_customer_date

df["order_delivered_customer_date"] = (
    df["order_delivered_customer_date"].replace({
        "2018-02-22 007:55": "2018-02-22 00:07:55",
        "2018-06-18:0 23:07:33": "2018-06-18 23:07:33"
    })
)

# Convert date columns to datetime

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="%Y-%m-%d %H:%M:%S"
    )

# Validate data types
display(df.dtypes)

# Check missing values after conversion
display(df[date_cols].isna().sum())

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

#### Data Types conclusion :

**Investigation:** The data types of all columns were inspected. The timestamp columns were initially stored as strings.

**Cleaning Decision:** The timestamp columns were converted to `datetime`. Two incorrectly formatted values in `order_delivered_customer_date` were corrected before conversion.

**Validation:** Data types and missing-value counts were checked after conversion to ensure the changes were applied correctly.

### invalid & suspicious values :

In [1270]:
df = orders_clean

#checking invalid values in order_status
display(df["order_status"].value_counts(dropna=False))

#after runnig the cell there is no invalid values

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [1271]:
#check chronological inconsistencies

checks = {
    "purchase_after_approval": (
        "order_purchase_timestamp",
        "order_approved_at"
    ),
    "purchase_after_carrier_delivery": (
        "order_purchase_timestamp",
        "order_delivered_carrier_date"
    ),
    "purchase_after_customer_delivery": (
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ),
    "approval_after_carrier_delivery": (
        "order_approved_at",
        "order_delivered_carrier_date"
    ),
    "approval_after_customer_delivery": (
        "order_approved_at",
        "order_delivered_customer_date"
    ),
    "customer_delivery_before_carrier_delivery": (
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    )
}

results = []

for name, (date1, date2) in checks.items():

    mask = (
        df[date1].notna()
        & df[date2].notna()
        & (df[date1] > df[date2])
    )

    results.append({
        "check": name,
        "records": mask.sum(),
        "percentage": round(mask.mean() * 100, 3)
    })

anomaly_summary = pd.DataFrame(results)

display(anomaly_summary)

,check,records,percentage
0,purchase_after_approval,0,0.000
1,purchase_after_carrier_delivery,166,0.167
2,purchase_after_customer_delivery,0,0.000
3,approval_after_carrier_delivery,1359,1.367
4,approval_after_customer_delivery,61,0.061
5,customer_delivery_before_carrier_delivery,23,0.023


In [1272]:
#fixing invalid values

# 1. Carrier date before purchase date
condition1 = (
    df["order_delivered_carrier_date"].notna()
    & df["order_purchase_timestamp"].notna()
    & (
        df["order_delivered_carrier_date"]
        < df["order_purchase_timestamp"]
    )
)

# 2. Customer delivery date before carrier date
condition2 = (
    df["order_delivered_customer_date"].notna()
    & df["order_delivered_carrier_date"].notna()
    & (
        df["order_delivered_customer_date"]
        < df["order_delivered_carrier_date"]
    )
)

# 3. Approval date after customer delivery date
condition3 = (
    df["order_approved_at"].notna()
    & df["order_delivered_customer_date"].notna()
    & (
        df["order_approved_at"]
        > df["order_delivered_customer_date"]
    )
)

#turn invalid values to NaT
df.loc[condition1, "order_delivered_carrier_date"] = pd.NaT
df.loc[condition2, "order_delivered_carrier_date"] = pd.NaT
df.loc[condition3, "order_approved_at"] = pd.NaT

# 4. Approval date after carrier date
# create anomaly column to declare the situation 
df["anomaly_approval_after_carrier"] = (
    df["order_approved_at"].notna()
    & df["order_delivered_carrier_date"].notna()
    & (
        df["order_approved_at"]
        > df["order_delivered_carrier_date"]
    )
)

In [1273]:
#validation
checks = {
    "carrier_before_purchase": df["order_delivered_carrier_date"] < df["order_purchase_timestamp"],
    "customer_before_carrier": df["order_delivered_customer_date"] < df["order_delivered_carrier_date"],
    "approval_after_delivery": df["order_approved_at"] > df["order_delivered_customer_date"],
    "approval_after_carrier_delivery": df["order_approved_at"] > df["order_delivered_carrier_date"]
}

for name, condition in checks.items():
    print(f"{name}: {condition.fillna(False).sum()} invalid records")

carrier_before_purchase: 0 invalid records
customer_before_carrier: 0 invalid records
approval_after_delivery: 0 invalid records
approval_after_carrier_delivery: 1132 invalid records


#### Timestamp Anomalies

The following decisions were made to handle chronological inconsistencies in the `orders` dataset.

| Issue | Count | Severity | Decision |
|---|---:|---|---|
| Carrier before purchase | 166 | Impossible | Set `order_delivered_carrier_date` to `NaT`. |
| Customer delivery before carrier | 23 | Impossible | Set `order_delivered_carrier_date` to `NaT`, keeping the customer delivery date unchanged. |
| Approval after customer delivery | 61 | Impossible | Set `order_approved_at` to `NaT`. |
| Approval after carrier | 1,359 | Mild | Keep both dates and flag the affected records. Exclude them only from relevant duration calculations. |

**Reasoning**

- **Carrier before purchase:** This sequence is chronologically impossible under the expected order process. The carrier date is therefore considered unreliable and set to `NaT`, while the order itself is retained.
- **Customer delivery before carrier:** Since the customer delivery date is retained as the more reliable event, the carrier date is set to `NaT`. This decision affects only 23 records.
- **Approval after customer delivery:** This sequence contradicts the expected order process. The approval timestamp is considered unreliable and set to `NaT`, while the customer delivery date is retained.
- **Approval after carrier:** This inconsistency may result from a recording delay rather than an incorrect event timestamp. Because the cause cannot be established confidently, both dates are retained and the affected records are flagged. They are excluded only from calculations of the duration between approval and carrier.

**Important:** No orders are deleted as part of this cleaning step. Only the selected timestamp values are changed to `NaT`; the remaining data are preserved.

In [1274]:
#check date range

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

display(df[date_cols].agg(["min", "max"]))

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12


#### Invalid & Suspicious Values conclusion

**Investigation:** The `order_status` values were checked for unexpected categories, and timestamp columns were examined for chronological inconsistencies between order purchase, approval, carrier handover, and customer delivery events. The date ranges were also inspected to understand the temporal coverage of the dataset.

**Findings:** Several timestamp inconsistencies were identified, where recorded events did not follow the expected chronological order.

**Decision:** The affected records were handled according to the nature and severity of each inconsistency. Timestamp values considered unreliable were marked as missing, while ambiguous cases were retained and flagged for use in relevant analyses.

**Action:** The selected timestamp values were set to `NaT`. No order records were deleted, and the remaining timestamps were preserved.

**Validation:** The chronological checks were used to identify the affected records and verify the results of the cleaning operations. The dataset was retained for further analysis, with flagged records excluded only from relevant duration calculations where necessary.

### preparation

In [1275]:

# 1. Extract order date components
df["order_year"] = df["order_purchase_timestamp"].dt.year
df["order_month"] = df["order_purchase_timestamp"].dt.month
df["order_day"] = df["order_purchase_timestamp"].dt.day
df["order_weekday"] = df["order_purchase_timestamp"].dt.day_name()



#finding Delivery time
df["delivery_duration_days"] = (
    df["order_delivered_customer_date"]
    - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400


#Number of days delayed in delivery
df["delivery_delay_days"] = (
    df["order_delivered_customer_date"]
    - df["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

####  Order Dates and Delivery Metrics

**Investigation:** The order purchase timestamp was used to extract date-related features. Additional metrics were created to measure delivery duration and compare actual delivery dates with estimated delivery dates.

**Transformation:** The following columns were added:
- `order_year`: Year of the order purchase.
- `order_month`: Month of the order purchase.
- `order_day`: Day of the month when the order was purchased.
- `order_weekday`: Name of the weekday when the order was purchased.
- `delivery_duration_days`: Time elapsed between purchase and customer delivery, measured in days.
- `delivery_delay_days`: Difference between the actual customer delivery date and the estimated delivery date, measured in days.

**Interpretation of `delivery_delay_days` column:**
- **Positive value:** Delivery was later than estimated.
- **Zero value:** Delivery was made on the estimated date.
- **Negative value:** Delivery was made earlier than estimated.
- **NaN value:** No information is available for comparison.

**Handling Missing Values:** Missing timestamps were preserved. Therefore, delivery metrics remain missing when the required dates are unavailable.

**Validation:** The new columns were inspected to verify their values and missing-value counts.

**Conclusion:** These features provide a foundation for analyzing order trends over time, weekday purchasing patterns, delivery duration, and delivery performance against estimated delivery dates.

### validation after cleaning :

In [1276]:
# 1. Table dimensions
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# 2. Duplicate records
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order IDs:", df["order_id"].duplicated().sum())

# 3. Missing values
print("\nMissing values per column:")
display(df.isna().sum().to_frame("Missing Values"))

# 4. Data types
print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

# 5. Chronological validation
checks = {
    "Purchase after approval": (
        df["order_purchase_timestamp"] > df["order_approved_at"]
    ),
    "Carrier before purchase": (
        df["order_delivered_carrier_date"] < df["order_purchase_timestamp"]
    ),
    "Customer delivery before carrier": (
        df["order_delivered_customer_date"] < df["order_delivered_carrier_date"]
    ),
    "Approval after customer delivery": (
        df["order_approved_at"] > df["order_delivered_customer_date"]
    ),
}

print("\nRemaining chronological inconsistencies:")
for name, condition in checks.items():
    print(f"{name}: {condition.fillna(False).sum()}")

# 6. Check flagged anomalies
if "anomaly_approval_after_carrier" in df.columns:
    print(
        "\nApproval after carrier (flagged):",
        df["anomaly_approval_after_carrier"].sum()
    )

Rows: 99441
Columns: 15
Duplicate rows: 0
Duplicate order IDs: 0

Missing values per column:


,Missing Values
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,221
order_delivered_carrier_date,1972
order_delivered_customer_date,2965
order_estimated_delivery_date,0
anomaly_approval_after_carrier,0
order_year,0



Data types:


,Data Type
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]
anomaly_approval_after_carrier,bool
order_year,int32



Remaining chronological inconsistencies:
Purchase after approval: 0
Carrier before purchase: 0
Customer delivery before carrier: 0
Approval after customer delivery: 0

Approval after carrier (flagged): 1132


### Data Cleaning Summary — Orders

**Overview:** The `orders` table was cleaned and prepared for further analysis. The process focused on data quality, consistency, and reliability while preserving the original order records.

**Key Activities:** The cleaning process included examining missing values, correcting data types, investigating invalid and suspicious timestamps, and identifying duplicate records. Additional date-related features and delivery metrics were created to support subsequent analysis.

**Data Integrity:** Missing values were retained where appropriate, and chronological inconsistencies were handled according to their severity. Records with ambiguous timestamp inconsistencies were preserved and flagged rather than arbitrarily modified or removed.

**Validation:** Final checks were performed on the table dimensions, duplicate records, data types, missing values, and remaining chronological inconsistencies to assess the results of the cleaning process.

**Conclusion:** The `orders` table is prepared for exploratory data analysis and further business analysis. Although the main cleaning steps have been completed, some timestamp inconsistencies and missing values remain by design and should be considered when performing analyses involving order timelines and delivery performance.

In [1277]:
#save orders_clean csv
df = orders_clean
df.to_csv("../Data/Processed/orders_clean.csv", index=False)

## customers Table :

### missing values

In [1278]:
df = customers_clean

#missing overview
missing_overview =pd.DataFrame({
    "missing_count" : df.isna().sum(),
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)

,missing_count,missing_percent
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


 **Decision :** there is no missing value in this table for now so we do nothing

### duplicates :

In [1279]:
df = customers_clean

#checking total duplicates
print(f"Exact duplicates: {df.duplicated().sum()}")

#check customer_id duplicates
print(f"customer_id duplicates: {df["customer_id"].duplicated().sum()}")

Exact duplicates: 0
customer_id duplicates: 0


**Decision** : there is no duplicates in whole table and customer_id column.

### Data types :

In [1280]:
df = customers_clean

display(df.dtypes)

display(df["customer_zip_code_prefix"].head())

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

0    14409
1     9790
2     1151
3     8775
4    13056
Name: customer_zip_code_prefix, dtype: int64

In [1281]:
#convert zip_code_prefix type to string
df["customer_zip_code_prefix"] = df["customer_zip_code_prefix"].astype("string").str.zfill(5)

#validation
print(df["customer_zip_code_prefix"].dtype)

display(df["customer_zip_code_prefix"].head())

string


0    14409
1    09790
2    01151
3    08775
4    13056
Name: customer_zip_code_prefix, dtype: string

#### Data Types conclusion

**Investigation:** The data types of the columns were inspected. The `customer_zip_code_prefix` column was stored as an integer, causing leading zeros in some postal codes to be removed.

**Decision:** The column was converted to string to preserve the original postal code format and treat postal codes as identifiers rather than numerical values.

**Action:** The values were converted to strings and padded with leading zeros to ensure a five-character format.

**Validation:** The column's data type and sample values were checked to confirm that leading zeros were preserved.

### invalid and suspicious values :

In [1282]:
df = customers_clean

#check customer_city unique value
print("unique number of cities:",df["customer_city"].nunique())
display(df["customer_city"].value_counts())

#check customer_state unique value
print("\nunique number of states:",df["customer_state"].nunique())
display(df["customer_state"].unique())

unique number of cities: 4119


customer_city
sao paulo              15540
rio de janeiro          6882
belo horizonte          2773
brasilia                2131
curitiba                1521
                       ...  
siriji                     1
natividade da serra        1
monte bonito               1
sao rafael                 1
eugenio de castro          1
Name: count, Length: 4119, dtype: int64


unique number of states: 27


<StringArray>
['SP', 'SC', 'MG', 'PR', 'RJ', 'RS', 'PA', 'GO', 'ES', 'BA', 'MA', 'MS', 'CE',
 'DF', 'RN', 'PE', 'MT', 'AM', 'AP', 'AL', 'RO', 'PB', 'TO', 'PI', 'AC', 'SE',
 'RR']
Length: 27, dtype: str

In [1283]:
city = df["customer_city"]

suspicious_city = (
    city.isna()
    | city.str.strip().eq("")
    | city.ne(city.str.strip())
)

print("Suspicious city values:", suspicious_city.sum())

display(
    df.loc[
        suspicious_city,
        ["customer_id", "customer_city", "customer_state"]
    ].head(10)
)

Suspicious city values: 0


,customer_id,customer_city,customer_state


In [1284]:
#Check the relationship with the order table
missing_customers = ~orders_clean["customer_id"].isin(customers_clean["customer_id"])

print("Orders without matching customer:", missing_customers.sum())

Orders without matching customer: 0


#### invalid and suspicious values conclusion

**Investigation:** unique values of customer city and state were checked and examined suspicious city. then we checked relationship of customer_id in orders table with customers table

**Decision:** we found no invalid or suspicious values and everything were normal, and also all customers_id in orders table were matched with customers table .


### validation :

In [1285]:
df = customers_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate customer_id:", df["customer_id"].duplicated().sum())

# 3. Postal code format
invalid_zip = ~df["customer_zip_code_prefix"].str.fullmatch(
    r"\d{5}", na=False
)
print("Invalid postal code prefixes:", invalid_zip.sum())

# 4. Data types
print("\nData types:")
print(df.dtypes)

Table shape: (99441, 5)
Total missing values: 0
Exact duplicate rows: 0
Duplicate customer_id: 0
Invalid postal code prefixes: 0

Data types:
customer_id                    str
customer_unique_id             str
customer_zip_code_prefix    string
customer_city                  str
customer_state                 str
dtype: object


### Data Cleaning Summary — Customers

**Overview:** The `customers` table was inspected and prepared for further analysis, focusing on data completeness, uniqueness, data types, and consistency.

**Key Activities:** Missing values and duplicate records were checked. The postal code prefix column was converted to string to preserve leading zeros. City and state values were examined, and the relationship between customer records and the `orders` table was validated.

**Validation:** The initial checks identified no missing values, exact duplicate rows, duplicate `customer_id` values, or suspicious city entries. All customer IDs referenced in the `orders` table had matching records in the `customers` table.

**Conclusion:** The `customers` table is ready for further analysis and integration with related tables. Final validation checks are used to confirm the postal code format, state codes, and overall data consistency.

In [1286]:
#save customers_clean csv
df = customers_clean
df.to_csv("../Data/Processed/customers_clean.csv", index=False)

## products Table :

### missing values:

In [1287]:
df = products_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum(),
    "missing_percent" : df.isna().mean() * 100
})

missing_overview = missing_overview[missing_overview["missing_count"] > 0].sort_values(by="missing_percent",ascending=False)

display(missing_overview)

,missing_count,missing_percent
product_category_name,610,1.851234
product_name_lenght,610,1.851234
product_description_lenght,610,1.851234
product_photos_qty,610,1.851234
product_weight_g,2,0.006070
product_length_cm,2,0.006070
product_height_cm,2,0.006070
product_width_cm,2,0.006070


In [1288]:
#investigation
# Check whether missing values occur in the same rows
missing_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

print("rows where all columns are missing:")
print(df[missing_columns].isna().all(axis=1).sum())

#check physical missing column occur in the same rows
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print(f"\nrows where all physical columns are missing: {df[physical_columns].isna().all(axis=1).sum()}")
display(df[df[physical_columns].isna().all(axis=1)])

#check order_items of product with no information
print("\nnumber of orders with this product that has no information except id :")
display(order_items_clean.loc[order_items_clean["product_id"] == "5eb564652db742ff8f28759cd8d2652a", "order_id"].count())

rows where all columns are missing:
610

rows where all physical columns are missing: 2


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



number of orders with this product that has no information except id :


np.int64(17)

#### conclusion

**Investigation :** we took a summary for missing values, The products dataset contained missing values in both descriptive and physical attributes. but after investigation we figured out each of them had missing in a same row and also a row has no information except `product_id`.

**Decision :** these products are related to orders in **order_items** table, so we couldn't drop rows with missing (even though that column has no information, because we found out 17 orders placed with this product ). This preserves the relationship between products and order items.


### duplicates :

In [1289]:
df = products_clean

#checking total duplicates
print("Exact duplicates:", df.duplicated().sum())

#check product_id duplicates
print("Duplicate product_id:",df["product_id"].duplicated().sum())

Exact duplicates: 0
Duplicate product_id: 0


**conclusion :** there is no duplicates in whole table and product_id column

### Data types :

In [1290]:
df = products_clean

display(df.dtypes)

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

**conclusion :** all data types are fine

### invalid values :

In [1291]:
#check values range

numerical_cols = ["product_name_lenght",
                  "product_description_lenght",
                  "product_photos_qty",
                  "product_weight_g" ,
                  "product_length_cm",
                  "product_height_cm",
                  "product_width_cm"]

display(df[numerical_cols].agg([min,max]))

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
min,5.0,4.0,1.0,0.0,7.0,2.0,6.0
max,76.0,3992.0,20.0,40425.0,105.0,105.0,118.0


**conclusion :** all of them are in proper range

### prepration

In [1292]:
df = products_clean
# Merge the English names onto products
df = df.merge(
    product_name_translation,
    on="product_category_name",
    how="left"
)

# Add the two missing translations manually
manual = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_and_food_preparers",
}
df["product_category_name"] = (
    df["product_category_name_english"]
    .fillna(df["product_category_name"].map(manual))
)

# Replace the Portuguese column with the English one
df = df.drop(columns="product_category_name_english") 

display(df["product_category_name"].value_counts())

product_category_name
bed_bath_table               3029
sports_leisure               2867
furniture_decor              2657
health_beauty                2444
housewares                   2335
                             ... 
fashion_childrens_clothes       5
home_comfort_2                  5
pc_gamer                        3
security_and_services           2
cds_dvds_musicals               1
Name: count, Length: 73, dtype: int64

In [1293]:
#validation
print(df.shape)  # should still be (32951, 9) after replacing
print(df["product_category_name"].isna().sum())

(32951, 9)
610


#### Result :

The `product_category_name` column in the `products` table was in Portuguese, so I translated it to English using the `product_category_name_translation` table.

The row count is unchanged (32,951), so no duplicates were introduced.

The 610 products with no category remain missing and will be handled in the cleaning stage.

Two categories (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`) have no entry in the translation table, so their English column is missing (13 products). we had to add them manually

### Products Table — Data Cleaning Summary

The `products` dataset contains **32,951 rows and 9 columns**, including product identifiers, category names, descriptive attributes, and physical measurements.

The missing-value investigation identified **610 products** with missing values in `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty`. These missing values occur in the same rows. Additionally, two products have missing values across all four physical measurement columns. One of these products has no available information other than its `product_id` and is referenced by 17 rows in the `order_items` table.

To preserve data integrity and relationships between tables, **no product records were removed**. Missing values were retained because reliable replacement values were unavailable.

The duplicate checks identified **no exact duplicate rows and no duplicate `product_id` values**. The data types were considered appropriate, and the numerical attributes were checked and found to be within the expected ranges.

During preparation, product category names were translated from Portuguese into English using the category translation dataset. Two categories without available translations were handled manually. However, the 610 products without category information remained missing.

**Conclusion:** The cleaning process focused on preserving product records, checking data consistency, and translating category names where possible. Although the dataset was prepared for further analysis, missing descriptive and physical attributes remain limitations and should be considered in analyses that depend on this information.

In [1294]:
#save product_clean table
df = products_clean

df.to_csv("../Data/Processed/products_clean.csv", index=False)

## order_items Table :

### missing values :

In [1295]:
#missing overview

df = order_items_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum(),
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)

,missing_count,missing_percent
order_id,0,0.0
order_item_id,0,0.0
product_id,0,0.0
seller_id,0,0.0
shipping_limit_date,0,0.0
price,0,0.0
freight_value,0,0.0


**there is no missing values in the table**

### duplicates :

In [1296]:
df = order_items_clean

#checking total duplicates
print("Exact duplicates:", df.duplicated().sum())

#check ids duplicates
print("Duplicate order_id:",df["order_id"].duplicated().sum())
print("Duplicate product_id:",df["product_id"].duplicated().sum())
print("Duplicate seller_id:",df["seller_id"].duplicated().sum())

Exact duplicates: 0
Duplicate order_id: 13984
Duplicate product_id: 79699
Duplicate seller_id: 109555


**conclusion :** we don't have any duplicates row in total but we figured out that the id columns are not unique and all of them reffered to other tables

### data types :

In [1297]:
df = order_items_clean

display(df.dtypes)

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [1298]:
#convert limit_date type to datetime
df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"],
                                           format="%Y-%m-%d %H:%M:%S")

# Validate data types
display(df.dtypes)

# Check missing values after conversion
display(df["shipping_limit_date"].isna().sum())

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

np.int64(0)

**conclusion :** `shipping_limit_date` type were converted to datetime and proper format.

**validation :** after validation and checking missing values after converstion no missing values or wrong data types were observed

### invalid and suspicious values :

In [1299]:
#check ranges

df = order_items_clean

#check cost ranges
price_cols = ["price","freight_value"]

display(df[price_cols].agg([min,max]).T)

#check shipping_limit_date range
display(df["shipping_limit_date"].agg([min,max]))

,min,max
price,0.85,6735.00
freight_value,0.00,409.68


min   2016-09-19 00:15:34
max   2020-04-09 22:35:08
Name: shipping_limit_date, dtype: datetime64[us]

In [1300]:
#check zero freight_value
df = order_items_clean

zero_freight = df["freight_value"].eq(0)

print("Zero freight rows:", zero_freight.sum())

zero_freight = df[df["freight_value"] == 0]

print("Unique orders:", zero_freight["order_id"].nunique())
print("Unique products:", zero_freight["product_id"].nunique())
print("Unique sellers:", zero_freight["seller_id"].nunique())

#check relationship between zero shipping costs with sellers and products
display(zero_freight["product_id"].value_counts().head())
display(zero_freight["seller_id"].value_counts().head())


Zero freight rows: 383
Unique orders: 339
Unique products: 10
Unique sellers: 9


product_id
53b36df67ebb7c41585e8d54d6772e08    187
aca2eb7d00ea1a7b8ebd4e68314663af     98
422879e10f46682990de24d770e7f83d     56
7a10781637204d8d10485c71a6108a2e     27
f1c7f353075ce59d8a6f3cf58f419c9c      9
Name: count, dtype: int64

seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
Name: count, dtype: int64

**conclusion :** A total of 383 order items have zero freight value, belonging to 339 unique orders, 10 unique products, and 9 sellers. These records are highly concentrated among a small number of sellers and products. No negative freight values were found. This distribution suggests that zero freight values may reflect specific shipping arrangements rather than random data errors. Therefore, these records were retained without modification.

### validation

In [1301]:
### validation :
df = order_items_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Value ranges
print("\nInvalid prices (<= 0):", (df["price"] <= 0).sum())
print("Negative freight values:", (df["freight_value"] < 0).sum())
print("Zero freight values (kept):", (df["freight_value"] == 0).sum())



# 5. Referential integrity with related tables
print("\nOrphan order_id (not in orders):",
      (~df["order_id"].isin(orders_clean["order_id"])).sum())
print("Orphan product_id (not in products):",
      (~df["product_id"].isin(products_clean["product_id"])).sum())
print("Orphan seller_id (not in sellers):",
      (~df["seller_id"].isin(sellers_clean["seller_id"])).sum())


Table shape: (112650, 7)
Total missing values: 0
Exact duplicate rows: 0

Data types:
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

Invalid prices (<= 0): 0
Negative freight values: 0
Zero freight values (kept): 383

Orphan order_id (not in orders): 0
Orphan product_id (not in products): 0
Orphan seller_id (not in sellers): 0


### Data Cleaning Summary — Order Items

**Overview:** The `order_items` table (112,650 rows, 7 columns) was inspected and prepared for further analysis. The work focused on completeness, uniqueness, data types, and the plausibility of price, freight, and date values, while preserving all item records.

**Key Activities:**
- **Missing values:** No missing values were found in any column, so no action was needed.
- **Duplicates:** No exact duplicate rows were found. `order_id`, `product_id`, and `seller_id` repeat many times, which is expected: each is a foreign key, and one order can contain several items. The table has no single unique column, but an item is identified by the combination of `order_id` and `order_item_id`.
- **Data types:** `shipping_limit_date` was stored as a string and was converted to `datetime`. The conversion introduced no missing values. All other columns already had appropriate types.
- **Value ranges:** `price` ranges from 0.85 to 6,735.00, with no zero or negative prices. `freight_value` ranges from 0.00 to 409.68, with no negative values.
- **Zero freight values:** 383 items (339 orders, 10 products, 9 sellers) have a freight value of 0. They are concentrated in a few sellers and products, which suggests specific shipping arrangements (e.g., free shipping) rather than random errors. These records were retained unchanged.


**Data Integrity:** No rows were deleted and no values were modified, apart from the data type conversion. Relationships with the `orders`, `products`, and `sellers` tables were checked as part of the final validation.

**Validation:** Final checks covered table dimensions, missing values, exact duplicates, uniqueness of `order_id`, data types, price and freight ranges, referential integrity with related tables.

**Conclusion:** The `order_items` table is ready for exploratory analysis and for joining with `orders`, `products`, and `sellers`. The remaining limitations are the zero freight values and the out-of-period shipping limit dates, both retained and documented rather than altered.

In [1302]:
# save order_items_clean csv
df = order_items_clean
df.to_csv("../Data/Processed/order_items_clean.csv", index=False)

## geolocation table :

### missing values :

In [1303]:
df = geolocation_clean

#missing overview
missing_overview =pd.DataFrame({
    "missing_count" : df.isna().sum(),
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)


,missing_count,missing_percent
geolocation_zip_code_prefix,0,0.0
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_city,0,0.0
geolocation_state,0,0.0


no missing values were found

### data types

In [1304]:
df = geolocation_clean

display(df.dtypes)
display(df["geolocation_zip_code_prefix"].head())

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

0    1037
1    1046
2    1046
3    1041
4    1035
Name: geolocation_zip_code_prefix, dtype: int64

In [1305]:
#convert zip_code_prefix type to string
df["geolocation_zip_code_prefix"] = df["geolocation_zip_code_prefix"].astype("string").str.zfill(5)

#validation
print(df["geolocation_zip_code_prefix"].dtype)

display(df["geolocation_zip_code_prefix"].head())

string


0    01037
1    01046
2    01046
3    01041
4    01035
Name: geolocation_zip_code_prefix, dtype: string

#### Data Types conclusion

**Investigation:** The data types of the columns were inspected. The `goelocation_zip_code_prefix` column was stored as an integer, causing leading zeros in some postal codes to be removed.

**Decision:** The column was converted to string to preserve the original postal code format and treat postal codes as identifiers rather than numerical values.

**Action:** The values were converted to strings and padded with leading zeros to ensure a five-character format.

**Validation:** The column's data type and sample values were checked to confirm that leading zeros were preserved.

### duplicates :

In [1306]:
df = geolocation_clean

#checking total duplicates
print(f"Exact duplicates: {df.duplicated().sum()}")

#check customer_id duplicates
print(f"zip-code prefix duplicates: {df["geolocation_zip_code_prefix"].duplicated().sum()}")

Exact duplicates: 261831
zip-code prefix duplicates: 981148


In [1307]:
print("Before drop:", geolocation_clean.shape)

geolocation_clean = geolocation_clean.drop_duplicates()

print("After drop:", geolocation_clean.shape)
print("Exact duplicates:",geolocation_clean.duplicated().sum())

Before drop: (1000163, 5)
After drop: (738332, 5)
Exact duplicates: 0


#### Conclusion

The duplicate analysis identified 261,831 exact duplicate rows. These records were removed because they contained no additional information. However, duplicate ZIP code prefixes were retained, as a single postal code prefix can correspond to multiple geographic coordinates.

**Final decision:** Exact duplicate rows were removed, while repeated ZIP code prefixes were preserved to maintain the geographical information associated with each postal code.

### invalid & suspicious values :

In [1308]:
df = geolocation_clean

#check geolocation_city unique value
print("unique number of cities:",df["geolocation_city"].nunique())
display(df["geolocation_city"].value_counts())

#check geolocation_state unique value
print("\nunique number of states:",df["geolocation_state"].nunique())
display(df["geolocation_state"].unique())

unique number of cities: 8011


geolocation_city
sao paulo                     79927
rio de janeiro                35177
são paulo                     19718
belo horizonte                19474
curitiba                      11263
                              ...  
almirante tamandaré do sul        1
três palmeiras                    1
são valentim                      1
três arroios                      1
ciríaco                           1
Name: count, Length: 8011, dtype: int64


unique number of states: 27


<StringArray>
['SP', 'RN', 'AC', 'RJ', 'ES', 'MG', 'BA', 'SE', 'PE', 'AL', 'PB', 'CE', 'PI',
 'MA', 'PA', 'AP', 'AM', 'RR', 'DF', 'GO', 'RO', 'TO', 'MT', 'MS', 'RS', 'PR',
 'SC']
Length: 27, dtype: str

In [1309]:
city = df["geolocation_city"]

suspicious_city = (
    city.isna()
    | city.str.strip().eq("")
    | city.ne(city.str.strip())
)

print("Suspicious city values:", suspicious_city.sum())

display(
    df.loc[suspicious_city].head(10)
)

Suspicious city values: 1


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
670078,40243,-12.987814,-38.505742,salvador,BA


In [1310]:
#fix suspicious city
df["geolocation_city"] = df["geolocation_city"].str.strip()

#validation
city = df["geolocation_city"]

suspicious_city = (
    city.isna()
    | city.str.strip().eq("")
    | city.ne(city.str.strip())
)

print("Suspicious city values:", suspicious_city.sum())


Suspicious city values: 0


In [1311]:
#Standardization of cities
df = geolocation_clean

before = df["geolocation_city"].nunique()

df["geolocation_city"] = (
    df["geolocation_city"]
    .str.strip()
    .str.lower()
    .str.normalize("NFKD")                  # split letters from accents (ã -> a + ~)
    .str.encode("ascii", errors="ignore")   # drop the accent marks
    .str.decode("ascii")
    .str.replace(r"[^a-z0-9\s'\-]", " ", regex=True)  # remove odd symbols, keep ' and -
    .str.replace(r"\s+", " ", regex=True)   # collapse repeated spaces
    .str.strip()
)

after = df["geolocation_city"].nunique()
print(f"Unique cities: {before} -> {after}")

# Standardizing can create new exact duplicates, so remove them
print("Duplicates created:", df.duplicated().sum())
geolocation_clean = df.drop_duplicates().reset_index(drop=True)
print("Final shape:", geolocation_clean.shape)

Unique cities: 8010 -> 5962
Duplicates created: 17838
Final shape: (720494, 5)


#### conclusion

The `geolocation_city` column was standardized to improve data consistency and reduce variations in city names.

The cleaning process included:
- Removing leading and trailing whitespace.
- Converting all city names to lowercase.
- Normalizing Unicode characters and removing accent marks.
- Replacing consecutive whitespace characters with a single space.

After standardization, the number of unique city names was compared with the original count to evaluate the changes.

Finally, duplicate records were identified and removed to maintain data consistency. The cleaned dataset was saved as `geolocation_clean`.

### validation

In [1312]:
### validation :
df = geolocation_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Postal code format
invalid_zip = ~df["geolocation_zip_code_prefix"].str.fullmatch(r"\d{5}", na=False)
print("\nInvalid postal code prefixes:", invalid_zip.sum())

# 5. City standardization
city = df["geolocation_city"]
print("\nUnique cities:", city.nunique())
print("Empty or missing city values:", (city.isna() | city.eq("")).sum())
print("Cities not fully standardized (uppercase, accents, odd symbols, extra spaces):",
      (~city.str.fullmatch(r"[a-z0-9]+(?:[ '\-][a-z0-9]+)*", na=False)).sum())

# 6. State checks
print("Unique states:", df["geolocation_state"].nunique())


Table shape: (720494, 5)
Total missing values: 0
Exact duplicate rows: 0

Data types:
geolocation_zip_code_prefix     string
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Invalid postal code prefixes: 0

Unique cities: 5962
Empty or missing city values: 0
Cities not fully standardized (uppercase, accents, odd symbols, extra spaces): 5
Unique states: 27


### Data Cleaning Summary — Geolocation

**Overview:** The `geolocation` table (1,000,163 rows, 5 columns) was inspected and prepared for further analysis. The work focused on completeness, duplicates, data types, and the consistency of city and state values.

**Key Activities:**
- **Missing values:** No missing values were found in any column, so no action was needed.
- **Data types:** `geolocation_zip_code_prefix` was stored as an integer, which removed leading zeros from some postal codes. It was converted to string and padded to five characters, so postal codes are treated as identifiers rather than numbers.
- **Duplicates (first pass):** 261,831 exact duplicate rows were found and removed because they carried no additional information, leaving 738,332 rows. Repeated ZIP code prefixes were kept, since one prefix can map to several coordinates.
- **City standardization:** City names contained whitespace issues, accent variants, and case variants (e.g., `são paulo` and `sao paulo`). They were standardized by trimming whitespace, converting to lowercase, removing accents, replacing special characters, and collapsing repeated spaces. This matches the format already used in the `customers` table. The number of unique cities decreased from 8,011 to X.
- **Duplicates (second pass):** Standardization created additional exact duplicates, which were removed, leaving Y rows. The index was then reset.
- **State values:** 27 unique state codes were found, which matches the number of Brazilian federative units. No invalid values were observed.

**Data Integrity:** Only exact duplicate rows were removed. Apart from the ZIP code type conversion and city name standardization, no values were modified. Typos and non-accent spelling errors in city names were not corrected.

**Validation:** Final checks covered table dimensions, missing values, exact duplicates, data types, postal code format, city name standardization, the number of states, whether coordinates fall within Brazil's approximate bounds, and how many city names in `customers` and `sellers` are found in `geolocation`.

**Conclusion:** The `geolocation` table is ready to be linked to `customers` and `sellers` through the ZIP code prefix. Since a prefix can still have several coordinate records, it should be aggregated to one location per prefix (e.g., the mean latitude and longitude) before joining, to avoid multiplying rows. Remaining city name mismatches, if any, are caused by typos or non-standard entries rather than accent or case differences, so city-level analysis should rely on the ZIP prefix or the `customers` and `sellers` city fields.

In [1313]:
# save geolocation_clean csv
df = geolocation_clean
df.to_csv("../Data/Processed/geolocation_clean.csv", index=False)

## sellers table :

### missing values :

In [1314]:
df = sellers_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum() ,
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)

,missing_count,missing_percent
seller_id,0,0.0
seller_zip_code_prefix,0,0.0
seller_city,0,0.0
seller_state,0,0.0


No missing values were found in the sellers.

### duplicates :

In [1315]:
# Check for duplicated sellers
print("Exact duplicate rows:", df.duplicated().sum())
print(f"Number of Duplicated Sellers: {df["seller_id"].duplicated().sum()}")

Exact duplicate rows: 0
Number of Duplicated Sellers: 0


There are no duplicated sellers in the dataset.
Everything seems to be fine!

### data types :

In [1316]:
df = sellers_clean

display(df.dtypes)

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

In [1317]:
#convert zip_code_prefix type to string
df["seller_zip_code_prefix"] = df["seller_zip_code_prefix"].astype("string").str.zfill(5)

#validation
print(df["seller_zip_code_prefix"].dtype)

display(df["seller_zip_code_prefix"].head())

string


0    13023
1    13844
2    20031
3    04195
4    12914
Name: seller_zip_code_prefix, dtype: string

### invalid & suspicious values :

In [1318]:
df = sellers_clean

#check seller_city unique value
print("unique number of cities:",df["seller_city"].nunique())
display(df["seller_city"].value_counts())

#check seller_state unique value
print("\nunique number of states:",df["seller_state"].nunique())
display(df["seller_state"].unique())

unique number of cities: 611


seller_city
sao paulo                 694
curitiba                  127
rio de janeiro             96
belo horizonte             68
ribeirao preto             52
                         ... 
aparecida de goiania        1
bandeirantes                1
vitoria de santo antao      1
palotina                    1
leme                        1
Name: count, Length: 611, dtype: int64


unique number of states: 23


<StringArray>
['SP', 'RJ', 'PE', 'PR', 'GO', 'SC', 'BA', 'DF', 'RS', 'MG', 'RN', 'MT', 'CE',
 'PB', 'AC', 'ES', 'RO', 'PI', 'MS', 'SE', 'MA', 'AM', 'PA']
Length: 23, dtype: str

In [1319]:
#check Suspicious city
city = df["seller_city"]

suspicious_city = (
    city.isna()
    | city.str.strip().eq("")
    | city.ne(city.str.strip())
)

print("Suspicious city values:", suspicious_city.sum())

display(
    df.loc[suspicious_city].head(10)
)

Suspicious city values: 0


,seller_id,seller_zip_code_prefix,seller_city,seller_state


In [1320]:
#Standardization of cities
df = sellers_clean

before = df["seller_city"].nunique()

df["seller_city"] = (
    df["seller_city"]
    .str.strip()
    .str.lower()
    .str.normalize("NFKD")                  # split letters from accents (ã -> a + ~)
    .str.encode("ascii", errors="ignore")   # drop the accent marks
    .str.decode("ascii")
    .str.replace(r"[^a-z0-9\s'\-]", " ", regex=True)  # remove odd symbols, keep ' and -
    .str.replace(r"\s+", " ", regex=True)   # collapse repeated spaces
    .str.strip()
)

after = df["seller_city"].nunique()
print(f"Unique cities: {before} -> {after}")

# Standardizing can create new exact duplicates, so remove them
print("Duplicates created:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Final shape:", df.shape)

Unique cities: 611 -> 605
Duplicates created: 0
Final shape: (3095, 4)


### validation

In [1321]:
### validation :
df = sellers_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate seller_id:", df["seller_id"].duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Postal code format
invalid_zip = ~df["seller_zip_code_prefix"].str.fullmatch(r"\d{5}", na=False)
print("\nInvalid postal code prefixes:", invalid_zip.sum())

# 5. City standardization
city = df["seller_city"]
print("\nUnique cities:", city.nunique())
print("Empty or missing city values:", (city.isna() | city.eq("")).sum())
print("Cities not fully standardized (uppercase, accents, odd symbols, extra spaces):",
      (~city.str.fullmatch(r"[a-z0-9]+(?:[ '\-][a-z0-9]+)*", na=False)).sum())

# 6. State checks
print("Unique states:", df["seller_state"].nunique())

# 7. Relationship with other tables
print("\nSellers not found in order_items (informational):",
      (~df["seller_id"].isin(order_items_clean["seller_id"])).sum())
print("Seller ZIP prefixes not found in geolocation (informational):",
      (~df["seller_zip_code_prefix"].isin(geolocation_clean["geolocation_zip_code_prefix"])).sum())

Table shape: (3095, 4)
Total missing values: 0
Exact duplicate rows: 0
Duplicate seller_id: 0

Data types:
seller_id                    str
seller_zip_code_prefix    string
seller_city                  str
seller_state                 str
dtype: object

Invalid postal code prefixes: 0

Unique cities: 605
Empty or missing city values: 0
Cities not fully standardized (uppercase, accents, odd symbols, extra spaces): 4
Unique states: 23

Sellers not found in order_items (informational): 0
Seller ZIP prefixes not found in geolocation (informational): 7


### Data Cleaning Summary — Sellers

**Overview:** The `sellers` table (3,095 rows, 4 columns) was inspected and prepared for further analysis. The work focused on completeness, uniqueness, data types, and the consistency of city and state values.

**Key Activities:**
- **Missing values:** No missing values were found in any column, so no action was needed.
- **Duplicates:** No exact duplicate rows and no duplicate `seller_id` values were found, so `seller_id` is a valid primary key.
- **Data types:** `seller_zip_code_prefix` was stored as an integer, which removed leading zeros from some postal codes. It was converted to string and padded to five characters, so postal codes are treated as identifiers rather than numbers.
- **City standardization:** No empty or whitespace-padded city values were found. City names were still standardized by trimming whitespace, converting to lowercase, removing accents, replacing special characters, and collapsing repeated spaces. This matches the format used in the `customers` and `geolocation` tables. The number of unique cities decreased from 611 to 605. The step created no new duplicate rows.
- **State values:** 23 unique state codes were found, all valid Brazilian state codes. Sellers are present in 23 of the 27 federative units.

**Data Integrity:** No rows were deleted. The only modified values are the ZIP code type conversion and the city name standardization. Typos and non-standard city entries (if any) were not corrected.

**Validation:** Final checks covered table dimensions, missing values, exact and `seller_id` duplicates, data types, postal code format, city name standardization, the number of states, and the relationship with the `order_items` and `geolocation` tables.

**Conclusion:** The `sellers` table is ready for exploratory analysis and for joining with `order_items` (through `seller_id`) and `geolocation` (through the ZIP code prefix). Since `geolocation` can have several coordinates per prefix, it should be aggregated to one location per prefix before joining, to avoid multiplying rows.

In [1322]:
# save sellers_clean csv
df = sellers_clean
df.to_csv("../Data/Processed/sellers_clean.csv", index=False)

## payments table :

### missing values :

In [1323]:
df = payments_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum() ,
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)

,missing_count,missing_percent
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


No missing values were found!

### duplicates :

In [1324]:
# Duplicate rows
print("Exact duplicates:", df.duplicated().sum())

# Duplicate payment identifiers
print(f"Duplicate payment records: {df.duplicated(subset=["order_id", "payment_sequential"]).sum()}")

Exact duplicates: 0
Duplicate payment records: 0


No duplicated rows or payments were found. Repeated order_id values are normal because one order can have multiple payment records.

### data types

In [1325]:
df = payments.copy()
display(df.dtypes)

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

The data types seem to be well-defined.

### invalid & suspicious values :

In [1326]:
#check payment method unique
df["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [1327]:
#check not defiend values
display(df[df["payment_type"] == "not_defined"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [1328]:
#check order status of not defiend payment types
df[
    df["payment_type"] == "not_defined"
].merge(
    orders_clean[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,canceled
1,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,canceled
2,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,canceled


#### Investigation of Undefined Payment Types

Three records were found with `payment_type = 'not_defined'`. All three records had a `payment_value` of 0.0 and were associated with canceled orders.

This pattern suggests that these records may represent orders canceled before a payment method was recorded. Since there is no reliable evidence to determine the actual payment method, these records were retained without modification.

**Decision:** The three records were kept, and their original values were preserved.

In [1329]:
#check installment & value range
display(df[["payment_installments" , "payment_value"]].agg([min , max]).T)

,min,max
payment_installments,0.0,24.00
payment_value,0.0,13664.08


In [1330]:
#check zero payment_installments
display(df[df["payment_installments"] == 0])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [1331]:
#check all orders with this ids
target_orders = [
    "744bade1fcf9ff3f31d860ace076d422",
    "1a57108394169c0b47d8f876acc9ba2d"
]

display(df[df["order_id"].isin(target_orders)].sort_values(by=["order_id", "payment_sequential"]))

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [1332]:
df = payments_clean
df.loc[df["payment_installments"] == 0,"payment_installments"] = np.nan

#validation
display(df[df["order_id"].isin(target_orders)])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,NaN,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,NaN,129.94


#### conclusion

Two records were found with `payment_installments = 0`. Both were credit card payments with positive payment values.

Further investigation showed that each order had only one payment record in the dataset, and both records had `payment_sequential = 2`. Since no additional payment records were available, the actual installment counts could not be determined.

**Decision:** The two records were retained, but their zero installment values were treated as missing (`NaN`) because zero is not a valid installment count.

In [1333]:
#check zero payments(not defiend didn't consider)
zero_payments = ((df["payment_value"] == 0) & (df["payment_type"] != "not_defined"))

display(zero_payments.sum())
display(df.loc[zero_payments])

np.int64(6)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1.0,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1.0,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1.0,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1.0,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1.0,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1.0,0.0


In [1334]:
df.loc[zero_payments].merge(
    orders_clean[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,8bcbe01d44d147f901cd3192671144db,4,voucher,1.0,0.0,delivered
1,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1.0,0.0,shipped
2,6ccb433e00daae1283ccc956189c82ae,4,voucher,1.0,0.0,delivered
3,45ed6e85398a87c253db47c2d9f48216,3,voucher,1.0,0.0,delivered
4,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1.0,0.0,shipped
5,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1.0,0.0,delivered


#### conclusion

Six records were identified with `payment_value = 0` and a defined payment type. All six records used the `voucher` payment method.

Further investigation showed that these records belonged to five orders: four with a `delivered` status and one with a `shipped` status. All six records had `payment_installments = 1`.

The payment history of these orders revealed that they also contained other payment records, most of which had positive payment values. This suggests that the zero-value voucher records may be part of multiple-payment transactions rather than necessarily indicating data errors.

**Decision:** All six records were retained without modification. Their zero payment values were preserved because there was insufficient evidence to classify them as invalid or determine alternative values.

In [1335]:
#Checking if the sum of prices in an order matches its payment value:

payment_totals = df.groupby("order_id")["payment_value"].sum()

item_totals = (
    order_items_clean.assign(total= order_items_clean["price"] + order_items_clean["freight_value"])
    .groupby("order_id")["total"].sum()
)

comparison = pd.concat(
    [payment_totals, item_totals],
    axis=1,
    keys=["payment_total", "item_total"]
)

comparison["difference"] = (
    comparison["payment_total"] - comparison["item_total"]
)

mismatches = comparison[
    comparison["difference"].abs() > 0.01
].copy()

print("Payment > Item total:",
      (mismatches["difference"] > 0.01).sum())

print("Payment < Item total:",
      (mismatches["difference"] < -0.01).sum())

print("Total mismatches:", len(mismatches))
mismatches["abs_difference"] = (
    mismatches["difference"].abs()
)

display(
    mismatches["abs_difference"].describe()
)

Payment > Item total: 290
Payment < Item total: 88
Total mismatches: 378


count    378.000000
mean       8.650714
std       16.090462
min        0.010000
25%        0.020000
50%        3.850000
75%       10.295000
max      182.810000
Name: abs_difference, dtype: float64

In [1336]:
#most difference between total price and payment valu
display(comparison.loc[comparison["difference"].abs().nlargest(10).index])

,payment_total,item_total,difference
order_id,,,
ce6d150fb29ada17d2082f4847107665,1586.47,1403.66,182.81
6e5fe7366a2e1bfbf3257dba0af1267f,406.92,287.91,119.01
70b742795bc441e94a44a084b6d9ce7a,578.82,466.93,111.89
996c7e73600ad3723e8627ab7bef81e4,664.43,587.90,76.53
70b7e94ea46d3e8b5bc12a50186edaf0,274.84,213.15,61.69
bc2c82b0ef78d2252b6176d1972db7c9,303.02,242.01,61.01
af9ffff2ce6b3defd34fd4c78857a379,466.97,413.17,53.80
262118ce178bb3e4590a3adcf6d62e6b,126.12,177.74,-51.62
bfdb5bbb06458d600a33d61f5f287472,394.36,348.93,45.43


#### conclusion:

A consistency check was performed by comparing the total payment value of each order with the sum of its item prices and freight values.

The analysis identified **378 orders** with a difference greater than 0.01 between these two totals:

- **290 orders (76.7%)** had a payment total higher than the item total.
- **88 orders (23.3%)** had a payment total lower than the item total.

The absolute differences ranged from approximately 0.01 to 182.81, with a mean difference of 8.65 and a median difference of 3.85.

These discrepancies indicate inconsistencies between the recorded payment totals and the calculated order totals. However, the available data does not provide sufficient evidence to determine the exact cause of each discrepancy.

**Decision:** All records were retained without modification. No payment values, item prices, or freight values were adjusted, as there was insufficient evidence to identify which values were incorrect. These discrepancies were documented as potential data limitations and should be considered in analyses involving order revenue and payment amounts.

### validation:

In [1337]:
### validation :
df = payments_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate (order_id, payment_sequential) pairs:",
      df.duplicated(subset=["order_id", "payment_sequential"]).sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Payment types
print("\nPayment types:")
print(df["payment_type"].value_counts(dropna=False))

# 5. Value ranges
print("\nInstallments range:", df["payment_installments"].min(), "-", df["payment_installments"].max())
print("Zero installments (should be 0):", (df["payment_installments"] == 0).sum())
print("Negative payment values:", (df["payment_value"] < 0).sum())
print("Zero payment values (kept):", (df["payment_value"] == 0).sum())
print("not_defined payment type (kept):", (df["payment_type"] == "not_defined").sum())

# 6. Referential integrity with related tables
print("\nOrphan order_id (not in orders):",
      (~df["order_id"].isin(orders_clean["order_id"])).sum())
print("Orders without any payment record (informational):",
      (~orders_clean["order_id"].isin(df["order_id"])).sum())

# 7. Payment total vs item total per order (flag only)
payment_totals = df.groupby("order_id")["payment_value"].sum()
item_totals = (
    order_items_clean.assign(total=order_items_clean["price"] + order_items_clean["freight_value"])
    .groupby("order_id")["total"].sum()
)
diff = (payment_totals - item_totals).dropna()
print("\nOrders where payment total differs from item total (flagged):",
      (diff.abs() > 0.01).sum())

Table shape: (103886, 5)
Missing values per column:
payment_installments    2
dtype: int64
Exact duplicate rows: 0
Duplicate (order_id, payment_sequential) pairs: 0

Data types:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments    float64
payment_value           float64
dtype: object

Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Installments range: 1.0 - 24.0
Zero installments (should be 0): 0
Negative payment values: 0
Zero payment values (kept): 9
not_defined payment type (kept): 3

Orphan order_id (not in orders): 0
Orders without any payment record (informational): 1

Orders where payment total differs from item total (flagged): 378


### Data Cleaning Summary — Payments

**Overview:** The `payments` table (103,886 rows, 5 columns) was inspected and prepared for further analysis. The work focused on completeness, duplicates, data types, and the plausibility of payment types, installments, and payment values, while preserving all payment records.

**Key Activities:**
- **Missing values:** No missing values were found in any column.
- **Duplicates:** No exact duplicate rows and no duplicate (`order_id`, `payment_sequential`) pairs were found. Repeated `order_id` values are expected, because one order can have several payment records.
- **Data types:** All columns already had appropriate types, so no conversion was needed.
- **Payment types:** Five values were found: `credit_card` (76,795), `boleto` (19,784), `voucher` (5,775), `debit_card` (1,529), and `not_defined` (3).
- **Undefined payment type:** The 3 `not_defined` records all have a payment value of 0 and belong to canceled orders, which suggests the orders were canceled before a payment method was recorded. They were retained unchanged.
- **Zero installments:** 2 credit card records had `payment_installments = 0`, which is not a valid installment count. Each belongs to an order with only one payment record (with `payment_sequential = 2`), so the real value could not be determined. The records were kept and the zero values were set to missing (`NaN`).
- **Zero payment values:** 6 voucher records (5 orders: 4 delivered, 1 shipped) have a payment value of 0. These orders also contain other payment records, so the zeros may be part of multiple-payment transactions rather than errors. They were retained unchanged.
- **Payment vs. item totals:** The payment total of each order was compared with the sum of its item prices and freight values. 378 orders differ by more than 0.01: 290 (76.7%) have a higher payment total and 88 (23.3%) a lower one. The absolute difference has a median of 3.85, a mean of 8.65, and a maximum of 182.81. The cause cannot be determined from the available data, so no values were changed.

**Data Integrity:** No rows were deleted. The only modification is converting the 2 zero installment values to missing. Zero payment values, `not_defined` records, and total mismatches were retained and documented.

**Validation:** Final checks covered table dimensions, missing values, exact and (`order_id`, `payment_sequential`) duplicates, data types, payment types, installment and payment value ranges, referential integrity with `orders`, and the comparison of payment totals with item totals.

**Conclusion:** The `payments` table is ready for exploratory analysis and for joining with `orders` through `order_id`. Because an order can have several payment records, the table should be aggregated to one row per order (e.g., the sum of `payment_value`) before joining, to avoid multiplying rows. Revenue analyses should keep in mind the 378 orders whose payment total does not match the item total, and the 2 missing installment values.

In [1338]:
# save payments_clean csv
df = payments_clean
df.to_csv("../Data/Processed/payments_clean.csv", index=False)

## reviews table:

### missing values:

In [1340]:
#Missing Values overview

df = reviews_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum() ,
    "missing_percent" : df.isna().mean() * 100
})

missing_overview = missing_overview[missing_overview["missing_count"] > 0].sort_values(by="missing_percent",ascending=False)
display(missing_overview)

,missing_count,missing_percent
review_comment_title,87656,88.341530
review_comment_message,58247,58.702532


#### conclusion

The missing-value analysis identified missing values in two columns:

- `review_comment_title`: 87,656 missing values (88.34%).
- `review_comment_message`: 58,247 missing values (58.70%).

These columns contain optional customer feedback. Customers can submit a review score without providing a written title or comment. Therefore, missing values in these columns do not necessarily indicate data quality issues.

**Decision:** Missing values were retained as `NaN` rather than replaced with empty strings or artificial text, since they represent the absence of written feedback rather than invalid data.

### duplicates:

In [1345]:
#checking total duplicates
print("Exact duplicates:", df.duplicated().sum())

Exact duplicates: 0


In [ ]:
#checking review_id duplicates
display(df["review_id"].duplicated().sum())

np.int64(814)

In [ ]:
#Check if records with the same review_id are exactly the same or have different information:
duplicates = df[
    df["review_id"].duplicated(keep=False)
].sort_values("review_id")

duplicates.groupby("review_id").nunique().head()

,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
review_id,,,,,,
00130cbe1f9d422698c812ed8ded1919,2,1,0,1,1,1
0115633a9c298b6a98bcbe4eee75345f,2,1,0,0,1,1
0174caf0ee5964646040cd94e15ac95e,2,1,0,1,1,1
017808d29fd1f942d97e50184dfb4c13,2,1,0,0,1,1
0254bd905dc677a6078990aad3331a36,2,1,0,1,1,1


In [ ]:
#Number of records per Review ID
review_id_counts = (
    df.groupby("review_id")
    .size()
    .loc[lambda x: x > 1]
)

print("Duplicated review IDs:", len(review_id_counts))
print("Total records with duplicated IDs:", review_id_counts.sum())
print("Maximum occurrences:", review_id_counts.max())

Duplicated review IDs: 789
Total records with duplicated IDs: 1603
Maximum occurrences: 3


In [ ]:
#check which columns are actually different, except order_id.
duplicates = df[
    df["review_id"].duplicated(keep=False)
]

cols = [
    col for col in df.columns
    if col not in ["review_id", "order_id"]
]

variation = (
    duplicates.groupby("review_id")[cols]
    .nunique()
)

display(variation.max().sort_values(ascending=False))

review_score               1
review_comment_title       1
review_comment_message     1
review_creation_date       1
review_answer_timestamp    1
dtype: int64

In [ ]:
#Does each order_id have only one review_id or is it itself linked to multiple review IDs?
reviews_clean.groupby("order_id")["review_id"].nunique().value_counts().sort_index()

review_id
1    98126
2      543
3        4
Name: count, dtype: int64

#### Conclusion

The duplicate analysis identified **789 duplicated `review_id` values**, representing **1,603 records**. Each duplicated review ID appeared in either two or three records.

Further investigation revealed that, within each group of duplicated `review_id` values, all columns except `order_id` contained identical values. In other words, the same review information was associated with multiple orders.

The relationship between orders and reviews was also examined. The results showed that:

- **98,126 orders** were associated with one review ID.
- **543 orders** were associated with two different review IDs.
- **4 orders** were associated with three different review IDs.

These findings indicate that the relationship between orders and reviews is not strictly one-to-one in this dataset. Additionally, `review_id` cannot be considered a unique identifier in the current data.

**Decision:** No records were removed or modified. Since there was insufficient evidence to determine which records or order-review relationships were incorrect, all records were preserved to avoid potentially losing valid information. The duplicated review IDs and multiple-review relationships were documented as data limitations for further analysis.

### data types:

In [1346]:
df = reviews_clean
display(df.dtypes)

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

In [1347]:
# Convert date columns to datetime

date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="%Y-%m-%d %H:%M:%S"
    )

# Validate data types
display(df.dtypes)

# Check missing values after conversion
display(df[date_cols].isna().sum())

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

review_creation_date       0
review_answer_timestamp    0
dtype: int64

date columns converted to datetime

### invalid & suspicious values:

In [1359]:
#check rating range
display(df["review_score"].agg([min,max]).T)

min    1
max    5
Name: review_score, dtype: int64

the rating range is correct

In [1361]:
#Timestamp consistency
answer_before_creation = df["review_answer_timestamp"] < df["review_creation_date"]
print("Answer timestamp before creation date (flagged):", answer_before_creation.sum())

Answer timestamp before creation date (flagged): 0


### validation:

In [1360]:
### validation :
df = reviews_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values (optional comment fields) and duplicates
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate (review_id, order_id) pairs:",
      df.duplicated(subset=["review_id", "order_id"]).sum())
print("Duplicated review_id values (kept, flagged):", df["review_id"].duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Value ranges
print("\nReview score range:", df["review_score"].min(), "-", df["review_score"].max())
print("Invalid scores (not 1-5):", (~df["review_score"].between(1, 5)).sum())
print("Date range:", df["review_creation_date"].min(), "->", df["review_creation_date"].max())

# 5. Timestamp consistency (flag only)
answer_before_creation = df["review_answer_timestamp"] < df["review_creation_date"]
print("Answer timestamp before creation date (flagged):", answer_before_creation.sum())

# 6. Order-review relationship
reviews_per_order = df.groupby("order_id")["review_id"].nunique()
print("\nOrders with more than one review_id (flagged):", (reviews_per_order > 1).sum())

# 7. Referential integrity with orders
print("Orphan order_id (not in orders):",
      (~df["order_id"].isin(orders_clean["order_id"])).sum())
print("Orders without any review (informational):",
      (~orders_clean["order_id"].isin(df["order_id"])).sum())

Table shape: (99224, 7)
Missing values per column:
review_comment_title      87656
review_comment_message    58247
dtype: int64
Exact duplicate rows: 0
Duplicate (review_id, order_id) pairs: 0
Duplicated review_id values (kept, flagged): 814

Data types:
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

Review score range: 1 - 5
Invalid scores (not 1-5): 0
Date range: 2016-10-02 00:00:00 -> 2018-08-31 00:00:00
Answer timestamp before creation date (flagged): 0

Orders with more than one review_id (flagged): 547
Orphan order_id (not in orders): 0
Orders without any review (informational): 768


In [1362]:
# save reviews_clean csv
df = reviews_clean
df.to_csv("../Data/Processed/reviews_clean.csv", index=False)

### Data Cleaning Summary — Reviews

**Overview:** The `reviews` table (99,224 rows, 7 columns) was inspected and prepared for further analysis. The work focused on completeness, duplicates, data types, and the plausibility of review scores, while preserving all review records.

**Key Activities:**
- **Missing values:** `review_comment_title` has 87,656 missing values (88.34%) and `review_comment_message` has 58,247 (58.70%). These fields are optional, because a customer can leave a score without writing a comment, so the missing values do not indicate a data quality problem. They were kept as `NaN` rather than replaced with empty strings or artificial text.
- **Duplicates (exact rows):** No exact duplicate rows were found.
- **Duplicates (`review_id`):** 789 `review_id` values appear more than once, covering 1,603 records, with each ID appearing two or three times. Within each group, all columns except `order_id` are identical, so the same review is linked to several orders.
- **Order-review relationship:** 98,126 orders have one review ID, 543 have two, and 4 have three. The relationship between orders and reviews is therefore not strictly one-to-one, and `review_id` is not a unique identifier. There was not enough evidence to decide which links are incorrect, so all records were kept and the issue was documented.
- **Data types:** `review_creation_date` and `review_answer_timestamp` were stored as strings and were converted to `datetime`. The conversion introduced no missing values. All other columns already had appropriate types.
- **Value ranges:** `review_score` ranges from 1 to 5, as expected.

**Data Integrity:** No rows were deleted and no values were modified, apart from the data type conversion. Missing comments, duplicated `review_id` values, and orders with multiple reviews were retained and documented.

**Validation:** Final checks covered table dimensions, missing values, exact and `review_id` duplicates, data types, score range, timestamp consistency, the number of reviews per order, and referential integrity with the `orders` table.

**Conclusion:** The `reviews` table is ready for exploratory analysis and for joining with `orders` through `order_id`. Because some orders have more than one review, the table should be aggregated to one row per order (e.g., the mean `review_score` or the latest review) before joining, to avoid multiplying order rows. Text-based analysis should account for the high share of missing comments, and `review_id` should not be used as a primary key.